# proctor basis
for proctor et al. 2018 fig 4e, maize: crop area weighted growing season means of each climate term of their yield model, g3 and rcp4.5, 2051 to 2069, mpi esm members 1 to 3, from their replication package (zenodo 10.5281/zenodo.1313800, 24 gb) (figure 5)

In [ ]:
from pathlib import Path
import configparser
import numpy as np, pandas as pd, xarray as xr

data = Path("data")
rep = Path("../data/raw/proctor2018/replication")  # unzipped replication package
gs = rep / "data/total_effect_analysis/processed/growingSeasonClimStacks"
terms = {"tas": ["MPI_Effect", "MPI_GS_RCS1", "MPI_GS_RCS2"], "pr": ["MPI_Effect", "MPI_GS_RCS1"],
         "clt": ["MPI_Effect", "MPI_GS_RCS1"], "saod": ["MPI_Effect"]}

In [ ]:
def read_grd(path):
    # r raster native format: grd header and gri binary
    c = configparser.ConfigParser()
    c.read(path)
    g, d = c["georeference"], c["data"]
    nr, nc, nb = int(g["nrows"]), int(g["ncols"]), int(d["nbands"])
    dt = {"FLT8S": "<f8", "FLT4S": "<f4", "INT4S": "<i4", "INT2S": "<i2"}[d["datatype"]]
    a = np.fromfile(str(path)[:-4] + ".gri", dtype=dt).astype(float)
    a = a.reshape(nr, nb, nc).transpose(1, 0, 2) if d.get("bandorder", "BIL") == "BIL" else a.reshape(nb, nr, nc)
    a[(a <= float(d.get("nodatavalue", "nan")) * 0.999) | ~np.isfinite(a)] = np.nan
    return a


# weights: maize harvested area fraction (monfreda) interpolated to the mpi grid, times cos latitude
m = xr.open_dataset(rep / "data/total_effect_analysis/cropMasks/maize_HarvAreaYield_NetCDF/maize_AreaYieldProduction.nc")
m = m.maizeData.isel(time=0, level=0)
lat, lon = 90 - 1.875 * (np.arange(96) + 0.5), -180 + 1.875 * (np.arange(192) + 0.5)
w = m.interp(latitude=lat, longitude=lon, method="linear").fillna(0).values * np.cos(np.radians(lat))[:, None]


def wmean(a):
    # weighted mean over all layers and pixels, missing values dropped
    x = a.reshape(a.shape[0], -1)
    ww = np.broadcast_to(w.ravel(), x.shape)
    k = np.isfinite(x)
    return float((x[k] * ww[k]).sum() / ww[k].sum())


rows = []
for scen in ("G3", "rcp45"):
    for var, ts in terms.items():
        for k, t in enumerate(ts):
            v = np.mean([wmean(read_grd(gs / f"{t}_maize_{scen}_E{e}_{var}.grd")) for e in (1, 2, 3)])
            rows.append({"scenario": scen, "var": var, "term": k, "value": v})
pd.DataFrame(rows).to_csv(data / "proctor_fig4e_maize_basis.csv", index=False)